In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.model_selection import GridSearchCV

In [2]:
dataset = pd.read_csv('CKD.csv')

In [3]:
dataset.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.0,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.0,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.0,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.0,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.0,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes


In [4]:
dataset.shape

(399, 25)

In [5]:
dataset = pd.get_dummies(dataset,drop_first='True')
dataset = dataset.astype(int)

In [6]:
dataset.head()

,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2,76,3,0,148,57,3,137,4,12,...,0,0,0,0,0,0,1,1,0,1
1,3,76,2,0,148,22,0,137,4,10,...,1,0,0,0,0,0,1,0,0,1
2,4,76,1,0,99,23,0,138,4,12,...,1,0,0,0,0,0,1,0,0,1
3,5,76,1,0,148,16,0,138,3,8,...,1,0,0,0,0,0,1,0,1,1
4,5,50,0,0,148,25,0,137,4,11,...,1,0,0,0,0,0,1,0,0,1


In [7]:
dataset.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='object')

In [8]:
indep = dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes']]
dep = dataset['classification_yes']

In [9]:
X_train,X_test,y_train,y_test = train_test_split(indep,dep,test_size=1/3,random_state=42)

In [10]:
sc = StandardScaler()

In [11]:
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [12]:
1# SVM Classifier

param_grid1 = {'kernel':['rbf','linear','poly','sigmoid'],
             'gamma':['auto','scale'],
             'C':[10,100,1000,2000,3000]}

grid1 = GridSearchCV(SVC(probability = True),param_grid1,refit=True,verbose=3,n_jobs=-1,scoring = 'f1_weighted')
classifier1 = grid1.fit(X_train,y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


In [13]:
y_pred1 = grid1.predict(X_test)

In [14]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred1)

In [15]:
print(cm)

[[53  0]
 [ 2 78]]


In [16]:
from sklearn.metrics import f1_score
f1_macro = f1_score(y_test,y_pred1,average='weighted')
print('f1_macro for the best parameter{}:'.format(grid1.best_params_),f1_macro)

f1_macro for the best parameter{'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}: 0.9850064683509054


In [17]:
clf_report = classification_report(y_test,y_pred1)
print('The confusion matric:\n',clf_report)

The confusion matric:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98        53
           1       1.00      0.97      0.99        80

    accuracy                           0.98       133
   macro avg       0.98      0.99      0.98       133
weighted avg       0.99      0.98      0.99       133



In [18]:
roc_auc = roc_auc_score(y_test,grid1.predict_proba(X_test)[:,-1])

In [19]:
print(roc_auc )

0.9997641509433962


In [20]:
re = grid1.cv_results_

In [21]:
tabel = pd.DataFrame.from_dict(re)
tabel

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.008428,0.001331,0.007705,0.001851,10,auto,rbf,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}",1.000000,0.981217,1.0,1.000000,1.000000,0.996243,0.007513,3
1,0.008323,0.001477,0.009054,0.004299,10,auto,linear,"{'C': 10, 'gamma': 'auto', 'kernel': 'linear'}",0.963284,0.962573,1.0,0.981233,0.981014,0.977621,0.013837,25
2,0.017779,0.002220,0.006969,0.000322,10,auto,poly,"{'C': 10, 'gamma': 'auto', 'kernel': 'poly'}",1.000000,0.981217,1.0,0.961755,1.000000,0.988594,0.015265,21
3,0.007946,0.001498,0.005763,0.000414,10,auto,sigmoid,"{'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}",0.981569,1.000000,1.0,1.000000,1.000000,0.996314,0.007372,1
4,0.007464,0.000578,0.005758,0.000799,10,scale,rbf,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",1.000000,0.981217,1.0,1.000000,1.000000,0.996243,0.007513,3
5,0.005255,0.000232,0.005004,0.000303,10,scale,linear,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",0.963284,0.962573,1.0,0.981233,0.981014,0.977621,0.013837,25
6,0.010687,0.001220,0.004591,0.000373,10,scale,poly,"{'C': 10, 'gamma': 'scale', 'kernel': 'poly'}",1.000000,0.981217,1.0,0.961755,1.000000,0.988594,0.015265,21
7,0.005982,0.000585,0.006633,0.001517,10,scale,sigmoid,"{'C': 10, 'gamma': 'scale', 'kernel': 'sigmoid'}",0.981569,1.000000,1.0,1.000000,1.000000,0.996314,0.007372,1
8,0.006189,0.000546,0.007444,0.002389,100,auto,rbf,"{'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}",1.000000,0.981217,1.0,1.000000,1.000000,0.996243,0.007513,3
9,0.006795,0.001311,0.005784,0.000448,100,auto,linear,"{'C': 100, 'gamma': 'auto', 'kernel': 'linear'}",0.963284,0.962573,1.0,0.981233,0.981014,0.977621,0.013837,25


In [22]:
2# Random Forest



param_grid2 = {'criterion':['gini','entropy'],
              'max_features': ['auto','sqrt','log2'],
              'n_estimators':[10,50,80,100]} 



grid2 = GridSearchCV(RandomForestClassifier(), param_grid2, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
classifier2 = grid2.fit(X_train, y_train) 

Fitting 5 folds for each of 24 candidates, totalling 120 fits


C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
40 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
7 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\base.py"

In [23]:
ypred2 = grid2.predict(X_test)

In [24]:
cm = confusion_matrix(y_test,ypred2)
print(cm)
f1_macro = f1_score(y_test,ypred2,average='weighted')
print('f1_macro with best parameter:{}'.format(grid2.best_params_),f1_macro)
print('The Classification report:\n',clf_report)
roc_auc1 = roc_auc_score(y_test,grid2.predict_proba(X_test)[:,-1])

print(roc_auc1)

[[51  2]
 [ 2 78]]
f1_macro with best parameter:{'criterion': 'gini', 'max_features': 'sqrt', 'n_estimators': 80} 0.9699248120300752
The Classification report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98        53
           1       1.00      0.97      0.99        80

    accuracy                           0.98       133
   macro avg       0.98      0.99      0.98       133
weighted avg       0.99      0.98      0.99       133

0.999056603773585


In [25]:
re1 = grid2.cv_results_
table1 = pd.DataFrame.from_dict(re1)
table1

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001724,0.000106,0.000000,0.000000,gini,auto,10,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,17
1,0.001437,0.000298,0.000000,0.000000,gini,auto,50,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,17
2,0.001481,0.000101,0.000000,0.000000,gini,auto,80,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,17
3,0.001315,0.000066,0.000000,0.000000,gini,auto,100,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,17
4,0.028844,0.001553,0.006686,0.001000,gini,sqrt,10,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.981569,0.981217,0.981014,0.961755,0.962264,0.973564,0.009437,16
5,0.128541,0.007580,0.010551,0.001079,gini,sqrt,50,"{'criterion': 'gini', 'max_features': 'sqrt', ...",1.000000,1.000000,1.000000,0.961755,0.981014,0.988554,0.015284,4
6,0.190524,0.005849,0.013982,0.000453,gini,sqrt,80,"{'criterion': 'gini', 'max_features': 'sqrt', ...",1.000000,1.000000,1.000000,0.981014,0.981014,0.992406,0.009301,1
7,0.255550,0.009818,0.017536,0.001331,gini,sqrt,100,"{'criterion': 'gini', 'max_features': 'sqrt', ...",1.000000,1.000000,1.000000,0.961755,0.981014,0.988554,0.015284,4
8,0.026119,0.001432,0.005695,0.000174,gini,log2,10,"{'criterion': 'gini', 'max_features': 'log2', ...",1.000000,0.981217,1.000000,0.961755,0.981014,0.984797,0.014285,14
9,0.130412,0.005780,0.011677,0.001341,gini,log2,50,"{'criterion': 'gini', 'max_features': 'log2', ...",1.000000,1.000000,1.000000,0.961755,0.981014,0.988554,0.015284,4


In [26]:
#3 LogisticRegression
param_grid3 = {'solver':['newton-cg', 'lbfgs', 'liblinear', 'saga'],
             'penalty':['l2']} 

grid3 = GridSearchCV(LogisticRegression(), param_grid3, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
classifier3 = grid3.fit(X_train, y_train) 

Fitting 5 folds for each of 4 candidates, totalling 20 fits


C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [27]:
ypredLR = classifier3.predict(X_test)

In [28]:

cm = confusion_matrix(y_test,ypredLR)
print(cm)
f1_macro = f1_score(y_test,ypredLR,average='weighted')
print('f1_macro with best parameter{}:'.format(grid3.best_params_),f1_macro)
print('The Classification report:\n',clf_report)
roc_aucLR = roc_auc_score(y_test,grid3.predict_proba(X_test)[:,-1])
print(roc_aucLR)

[[52  1]
 [ 2 78]]
f1_macro with best parameter{'penalty': 'l2', 'solver': 'saga'}: 0.9774780806716137
The Classification report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98        53
           1       1.00      0.97      0.99        80

    accuracy                           0.98       133
   macro avg       0.98      0.99      0.98       133
weighted avg       0.99      0.98      0.99       133

0.9988207547169812


In [29]:
re3 =grid3.cv_results_
table3 = pd.DataFrame.from_dict(re3)

table3

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.086165,0.004550,0.016326,0.008119,l2,newton-cg,"{'penalty': 'l2', 'solver': 'newton-cg'}",0.981569,0.981217,1.000000,1.000000,1.000000,0.992557,0.009116,2
1,0.050277,0.032859,0.008759,0.001412,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.981569,0.981217,1.000000,1.000000,1.000000,0.992557,0.009116,2
2,0.005001,0.000655,0.006154,0.000657,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.963284,1.000000,0.981233,0.981233,0.981233,0.981397,0.011612,4
3,0.030055,0.002238,0.004050,0.000096,l2,saga,"{'penalty': 'l2', 'solver': 'saga'}",0.963284,1.000000,1.000000,1.000000,1.000000,0.992657,0.014687,1


In [31]:
#4 KNN algorthim

param_grid2 = {'n_neighbors':[4,6,8],
             'metric':['minkowski','sqeuclidean', 'euclidean','hamming', 'russellrao', 'canberra'],'p':[2]} 

grid4 = GridSearchCV(KNeighborsClassifier(), param_grid2, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
classifier4 = grid4.fit(X_train, y_train) 

Fitting 5 folds for each of 18 candidates, totalling 90 fits


In [32]:
ypred = grid4.predict(X_test)

In [33]:
cm = confusion_matrix(y_test,ypred)
print(cm)
f1_macro = f1_score(y_test,ypred,average='weighted')
print('The f1_macro with best_params{}:'.format(grid4.best_params_),f1_macro)
print('The Classification report:\n',clf_report)
roc_auc = roc_auc_score(y_test,grid4.predict_proba(X_test)[:,-1])
print(roc_auc)
re = grid4.cv_results_
table = pd.DataFrame.from_dict(re)
table


[[53  0]
 [ 8 72]]
The f1_macro with best_params{'metric': 'minkowski', 'n_neighbors': 4, 'p': 2}: 0.9403772589368158
The Classification report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98        53
           1       1.00      0.97      0.99        80

    accuracy                           0.98       133
   macro avg       0.98      0.99      0.98       133
weighted avg       0.99      0.98      0.99       133

0.9851415094339624


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbors,param_p,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002257,0.000458,0.131162,0.029305,minkowski,4,2,"{'metric': 'minkowski', 'n_neighbors': 4, 'p': 2}",0.981569,0.981217,1.000000,0.944161,0.944161,0.970222,0.022337,1
1,0.003660,0.001340,0.103186,0.079921,minkowski,6,2,"{'metric': 'minkowski', 'n_neighbors': 6, 'p': 2}",0.945100,0.981217,0.981233,0.962636,0.962636,0.966564,0.013575,4
2,0.002407,0.000089,0.009330,0.002998,minkowski,8,2,"{'metric': 'minkowski', 'n_neighbors': 8, 'p': 2}",0.908877,0.962573,0.962636,0.944161,0.962636,0.948177,0.020909,10
3,0.002281,0.000049,0.015302,0.001008,sqeuclidean,4,2,"{'metric': 'sqeuclidean', 'n_neighbors': 4, 'p...",0.981569,0.981217,1.000000,0.944161,0.944161,0.970222,0.022337,1
4,0.002092,0.000067,0.013125,0.000613,sqeuclidean,6,2,"{'metric': 'sqeuclidean', 'n_neighbors': 6, 'p...",0.945100,0.981217,0.981233,0.962636,0.962636,0.966564,0.013575,4
5,0.001961,0.000083,0.016635,0.006551,sqeuclidean,8,2,"{'metric': 'sqeuclidean', 'n_neighbors': 8, 'p...",0.908877,0.962573,0.962636,0.944161,0.962636,0.948177,0.020909,10
6,0.001877,0.000159,0.011650,0.000513,euclidean,4,2,"{'metric': 'euclidean', 'n_neighbors': 4, 'p': 2}",0.981569,0.981217,1.000000,0.944161,0.944161,0.970222,0.022337,1
7,0.001787,0.000128,0.011752,0.001125,euclidean,6,2,"{'metric': 'euclidean', 'n_neighbors': 6, 'p': 2}",0.945100,0.981217,0.981233,0.962636,0.962636,0.966564,0.013575,4
8,0.001609,0.000050,0.010126,0.000087,euclidean,8,2,"{'metric': 'euclidean', 'n_neighbors': 8, 'p': 2}",0.908877,0.962573,0.962636,0.944161,0.962636,0.948177,0.020909,10
9,0.001595,0.000092,0.022342,0.016791,hamming,4,2,"{'metric': 'hamming', 'n_neighbors': 4, 'p': 2}",0.926978,0.888515,0.981233,0.944161,0.944161,0.937010,0.030034,15


In [34]:
# 5 DecisionTree

param_grid = {'criterion':['gini','entropy'],
              'max_features': ['auto','sqrt','log2'],
              'splitter':['best','random']} 

grid5 = GridSearchCV(DecisionTreeClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
classifier5 = grid5.fit(X_train, y_train) 

Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
8 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\gopi\Anaconda\envs\myenv\Lib\site-packages\sklearn\base.py",

In [35]:
ypred = grid5.predict(X_test)

In [36]:
cm = confusion_matrix(y_test,ypred)
print(cm)
f1_macro = f1_score(y_test,ypred,average='weighted')
print('The f1_macro with best_params{}:'.format(grid5.best_params_),f1_macro)
print('The Classification report:\n',clf_report)
roc_auc = roc_auc_score(y_test,grid5.predict_proba(X_test)[:,-1])
print(roc_auc)
re = grid5.cv_results_
table = pd.DataFrame.from_dict(re)
table

[[51  2]
 [ 2 78]]
The f1_macro with best_params{'criterion': 'entropy', 'max_features': 'sqrt', 'splitter': 'random'}: 0.9699248120300752
The Classification report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98        53
           1       1.00      0.97      0.99        80

    accuracy                           0.98       133
   macro avg       0.98      0.99      0.98       133
weighted avg       0.99      0.98      0.99       133

0.9686320754716982


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001145,0.000022,0.000000,0.000000,gini,auto,best,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
1,0.001179,0.000158,0.000000,0.000000,gini,auto,random,"{'criterion': 'gini', 'max_features': 'auto', ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
2,0.004012,0.000874,0.009676,0.001432,gini,sqrt,best,"{'criterion': 'gini', 'max_features': 'sqrt', ...",1.000000,0.962573,0.962636,0.943041,0.925272,0.958704,0.024895,4
3,0.004239,0.000979,0.009733,0.002831,gini,sqrt,random,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.981569,0.981217,1.000000,0.906166,0.944161,0.962623,0.033556,3
4,0.004125,0.001143,0.008473,0.001183,gini,log2,best,"{'criterion': 'gini', 'max_features': 'log2', ...",0.945100,0.888286,0.906935,0.961755,0.924528,0.925321,0.026187,8
5,0.004724,0.002274,0.011242,0.003940,gini,log2,random,"{'criterion': 'gini', 'max_features': 'log2', ...",0.945100,1.000000,0.981233,0.962636,0.962264,0.970247,0.018761,2
6,0.001746,0.000494,0.000000,0.000000,entropy,auto,best,"{'criterion': 'entropy', 'max_features': 'auto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
7,0.001765,0.000250,0.000000,0.000000,entropy,auto,random,"{'criterion': 'entropy', 'max_features': 'auto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
8,0.004688,0.000275,0.009070,0.000792,entropy,sqrt,best,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.906891,0.962573,1.000000,0.943041,0.961755,0.954852,0.030283,5
9,0.004350,0.000636,0.008994,0.000439,entropy,sqrt,random,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.981569,0.962573,1.000000,0.962264,0.981233,0.977528,0.014083,1


In [ ]:
#SVM Model has better performance
import pickle

In [ ]:
filename = 'Finilize_model_SVC.sav'
pickle.dump(classifier1,open(filename, 'wb'))
pickle.dump(sc,open('Scaler_model.sav','wb'))

In [ ]:
preinput= sc.transform([[2,	76,	3,0,148,57,3,137,4,12,0	,0,	0,	0,	0,	0,	1,	1,	0 ,1,0,0,1,0,1,0,0]])

In [ ]:
load_model = pickle.load(open('Finilize_model_SVC.sav','rb'))

In [ ]:
result = load_model.predict(preinput)

In [ ]:
print(result)

In [ ]:
dataset.columns

In [ ]:
dataset.head()